# 4 — Stage 1: is this sentence an adverse drug event?

A yes/no decision on a whole sentence.

> *"A case of toxic hepatitis caused by methotrexate."* → **ADE**
> *"She was treated with cisplatin, vinblastine and bleomycin."* → **not ADE**

The second one mentions three drugs and is still not an ADE, because nothing bad is
reported. That is the difficulty in one line: this is not keyword spotting.

Eight models were trained, in a deliberate ladder from simplest to most powerful. Four of
them are the **embedding ablation** — the same network, four different starting vectors —
and that is the result the project exists to produce.

> ### About running this notebook
>
> **The training code below is real and complete** — it is the code that produced the
> checkpoints in `models/stage1/`. It sits behind a `TRAIN` switch that is **off** by
> default, because runs 3-8 need a GPU and a few hours.
>
> With `TRAIN = False` (the default) the notebook loads the finished checkpoints and
> scores them, which takes seconds on a laptop CPU. Set `TRAIN = True` on a Kaggle GPU
> session to retrain everything from scratch.

In [ ]:
import sys, json, textwrap
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
pd.set_option("display.max_colwidth", 90)
print("project root:", ROOT)

---

## 4.1 The data, and why accuracy is the wrong metric

In [ ]:
train = pd.read_parquet(ROOT / "data" / "splits" / "stage1_train.parquet")
test = pd.read_parquet(ROOT / "data" / "splits" / "stage1_test.parquet")

print(f"train {len(train):,} sentences, test {len(test):,} sentences\n")
print(train["label"].value_counts().rename({0: "not ADE", 1: "ADE"}).to_string())
print(f"\npositive rate: {train['label'].mean():.2%}")
print(f"always predicting 'not ADE' would score {1 - test['label'].mean():.1%} accuracy")

**79.6% accuracy for a model that has learned nothing.** So accuracy is reported but never
used to choose anything.

The primary metric is **macro-F1**: compute F1 for the ADE class and F1 for the not-ADE
class, then average them with equal weight. A model that ignores the minority class scores
badly on it and the average drops. The dev split selects the model; the test split is
scored once at the end.

---

## 4.2 The model ladder

| Tier | Runs | Model | Idea |
|---|---|---|---|
| T1 | 1 | Multinomial Naive Bayes | word counts, independence assumption |
| T2 | 2, 2b | Logistic Regression, Linear SVM | TF-IDF features, a learned linear boundary |
| T3 | **3-6** | **BiLSTM + attention** | **word order and context — the ablation** |
| T4 | 7 | BERT base | general-purpose transformer, fine-tuned |
| T5 | 8 | BiomedBERT | a transformer pretrained on PubMed |

Each tier tests one idea, and the jump between tiers is the interesting part.

The switch below controls every training cell in this notebook.

In [ ]:
# ---------------------------------------------------------------------------
# OFF by default. Runs 3-8 need a GPU and a few hours; the finished checkpoints
# are already in models/stage1/, and section 4.4 loads and scores them.
#
# Set to True in a Kaggle GPU session to retrain runs 1-8 from scratch.
# ---------------------------------------------------------------------------
TRAIN = False

from src.utils import DEFAULT_SEED, get_device_count, log_run, pin_single_gpu, set_seed

SPLITS = ROOT / "data" / "splits"
MATRICES = ROOT / "models" / "emb_matrices"
OUT = ROOT / "models" / "stage1"

print(f"TRAIN = {TRAIN}")
print(f"visible GPUs: {get_device_count()}")

### T1-T2: the sparse baselines

Fast — they run on a CPU in seconds. They exist so the neural results have a floor to beat:
a BiLSTM below TF-IDF logistic regression means the BiLSTM is undertrained, not that
embeddings do not help.

Two details that matter. The vectorizer is fitted on **train only** (fitting on train+test
leaks test vocabulary into the features), and `class_weight="balanced"` is what stops the
1:4 imbalance from producing a model that always says "not ADE". Hyperparameters are chosen
on **dev**; test is scored once, at the end, with the winner.

In [ ]:
def train_baselines():
    """Runs 1, 2 and 2b. sklearn, CPU, a few seconds each."""
    import time

    from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
    from sklearn.linear_model import LogisticRegression
    from sklearn.naive_bayes import MultinomialNB
    from sklearn.svm import LinearSVC

    from src.metrics import classification_metrics, macro_f1
    from src.tokenizer import tokenize
    from src.vocab import is_indexable

    def project_tokenizer(text):
        """tokenize(), minus punctuation.

        The filter matters: it is the same one the neural vocabulary was built
        with, so the sparse and dense models see an identical word stream. Leave
        it out and runs 1-2b are scored on different features from runs 3-8.
        """
        return [t for t in tokenize(text)[0] if is_indexable(t)]

    seed = set_seed(DEFAULT_SEED)
    splits = {s: pd.read_parquet(SPLITS / f"stage1_{s}.parquet")
              for s in ("train", "dev", "test")}
    ytr, ydev, yte = (splits[s]["label"].values for s in ("train", "dev", "test"))

    # token_pattern=None and lowercase=False because OUR tokenizer already does
    # both - sklearn's defaults would undo the abbreviation protection.
    shared = dict(tokenizer=project_tokenizer, lowercase=False,
                  ngram_range=(1, 2), min_df=2, token_pattern=None)
    count_vec = CountVectorizer(**shared)
    tfidf_vec = TfidfVectorizer(**shared, sublinear_tf=True)

    counts = [count_vec.fit_transform(splits["train"]["text"])] + \
             [count_vec.transform(splits[s]["text"]) for s in ("dev", "test")]
    tfidf = [tfidf_vec.fit_transform(splits["train"]["text"])] + \
            [tfidf_vec.transform(splits[s]["text"]) for s in ("dev", "test")]

    specs = [
        ("1", "naive_bayes", "count-1-2gram", MultinomialNB,
         [{"alpha": a} for a in (0.1, 0.3, 1.0, 3.0)], counts),
        ("2", "logreg", "tfidf-1-2gram",
         lambda **kw: LogisticRegression(max_iter=2000, random_state=seed, **kw),
         [{"C": c, "class_weight": w} for c in (0.5, 1.0, 4.0, 16.0)
          for w in (None, "balanced")], tfidf),
        ("2b", "linear_svm", "tfidf-1-2gram",
         lambda **kw: LinearSVC(random_state=seed, **kw),
         [{"C": c, "class_weight": w} for c in (0.1, 0.5, 1.0, 4.0)
          for w in (None, "balanced")], tfidf),
    ]

    for run_id, name, features, build_model, grid, (Xtr, Xdev, Xte) in specs:
        t0 = time.time()

        best = (-1.0, None, None)
        for params in grid:                       # selection happens on DEV
            model = build_model(**params).fit(Xtr, ytr)
            score = macro_f1(ydev, model.predict(Xdev))
            if score > best[0]:
                best = (score, params, model)
        dev_f1, params, model = best
        seconds = time.time() - t0

        scores = (model.predict_proba(Xte)[:, 1] if hasattr(model, "predict_proba")
                  else model.decision_function(Xte))
        metrics = classification_metrics(yte, model.predict(Xte), scores)
        metrics["dev_macro_f1"] = float(dev_f1)
        metrics["train_seconds"] = round(seconds, 2)

        print(f"run {run_id:>2s}  {name:12s} {params}")
        print(f"      dev {dev_f1:.4f} | TEST macro-F1 {metrics['macro_f1']:.4f} "
              f"| {seconds:.1f}s over {len(grid)} configs")

        log_run(run_id=run_id, stage="1", model=name, embedding=features,
                metrics=metrics,
                params={**{k: str(v) for k, v in params.items()},
                        "ngram_range": [1, 2], "min_df": 2,
                        "n_features": int(Xtr.shape[1]),
                        "tokenizer": "src.tokenizer.tokenize",
                        "selected_on": "dev macro_f1", "fit_on": "train only",
                        "grid_size": len(grid)},
                seed=seed, notes=f"Step 4.1 sparse baseline; sklearn, CPU, {seconds:.1f}s.")


if TRAIN:
    train_baselines()

### T3: the BiLSTM — the model the ablation runs on

```
word ids -> Embedding (E0/E1/E2/E3) -> BiLSTM -> attention -> dropout -> linear -> 2
```

Three design choices worth defending:

- **Bidirectional LSTM.** "The rash resolved after stopping the drug" and "The drug was
  stopped after the rash resolved" contain the same words. Order is the signal.
- **Additive attention instead of mean pooling.** An ADE sentence is usually positive
  because of a short span — *"developed hepatotoxicity"* — buried in a long clinical
  description. Attention can concentrate on a few positions; an average over 18 words
  dilutes them.
- **The embedding matrix is a constructor argument, not built inside.** This is what makes
  runs 3-6 a controlled experiment.

The definition is in `src/models.py`, and is built here so you can see its size.

In [ ]:
import numpy as np
from src.models import BiLSTMClassifier

matrix = np.load(MATRICES / "E3.npy")
model = BiLSTMClassifier(matrix, num_classes=2, hidden_dim=256, dropout=0.5,
                         freeze_embeddings=True)

print(model)
total = sum(p.numel() for p in model.parameters())
print(f"\nparameters total     : {total:,}")
print(f"of which trainable   : {model.trainable_parameters():,}")
print(f"frozen in the embedding: {total - model.trainable_parameters():,}"
      f"  ({matrix.shape[0]:,} words x {matrix.shape[1]})")

`freeze_embeddings=True` is why 3.7M of the 4.9M parameters get no gradient: the run is
measuring **the pretrained vectors themselves**, not what the task can teach them.

### The training loop

Ordinary supervised training. Nothing exotic — which is the point, since anything exotic
would be another variable between the four runs.

Three things in the code below are there for specific reasons:

- **`pin_single_gpu()` runs before torch is imported.** Kaggle gives you two T4s, and
  `DataParallel` would silently double the effective batch. If that happened to only some
  of runs 3-6, the ablation would be comparing batch sizes as well as embeddings. The cell
  raises rather than warns if more than one GPU is visible.
- **Gradient clipping at 5.0.** Exploding gradients are the standard LSTM failure, and
  without clipping a single bad batch can wipe an otherwise good run.
- **Early stopping on dev macro-F1, patience 3.** Test is touched exactly once, at the end,
  using the epoch dev chose.

In [ ]:
MAX_LEN = 96          # covers 100% of training sentences; p95 is 33 tokens
PATIENCE = 3

EMBEDDINGS = {"E0_random": "random init - the ablation floor",
              "E1": "GloVe 300d, general purpose",
              "E2": "our Word2Vec, PubMed",
              "E3": "our FastText, PubMed"}


def train_bilstm_run(run_id, embedding, *, unfreeze=False, epochs=15, batch_size=32,
                     lr=1e-3, hidden_dim=256, dropout=0.5, seed=DEFAULT_SEED,
                     dataset_version=""):
    """One run of the ablation. `embedding` is the ONLY thing that changes for 3-6."""
    import time

    import numpy as np
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset

    from src.metrics import classification_metrics, macro_f1
    from src.models import BiLSTMClassifier, class_weights
    from src.vocab import encode_batch, load_vocab, unk_rate

    set_seed(seed)
    device_count = get_device_count()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    if device_count > 1:
        # This has to stop the run, not warn: a warning scrolls past, and the
        # ablation would then be comparing batch sizes as well as embeddings.
        raise SystemExit(
            f"{device_count} GPUs visible, so the effective batch would be "
            f"{batch_size * device_count}, not {batch_size}. Runs 3-6 must all "
            "see the same device count. Call pin_single_gpu() before importing torch.")

    vocab, index = load_vocab(MATRICES / "vocab.json")
    matrix = np.load(MATRICES / f"{embedding}.npy")
    if matrix.shape[0] != len(vocab):
        raise ValueError(f"{embedding}.npy has {matrix.shape[0]} rows but vocab.json "
                         f"has {len(vocab)} - they must come from the same build.")

    loaders, frames = {}, {}
    for split in ("train", "dev", "test"):
        df = pd.read_parquet(SPLITS / f"stage1_{split}.parquet")
        ids, lengths = encode_batch(df["text"].tolist(), index, MAX_LEN)
        dataset = TensorDataset(torch.tensor(ids, dtype=torch.long),
                                torch.tensor(lengths, dtype=torch.long),
                                torch.tensor(df["label"].values, dtype=torch.long))
        # Only train is shuffled; dev and test keep parquet row order so saved
        # predictions line up with the split for the error analysis later.
        loaders[split] = DataLoader(dataset, batch_size=batch_size,
                                    shuffle=(split == "train"), num_workers=0)
        frames[split] = df

    rate, unks, total = unk_rate(frames["train"]["text"], index)
    print(f"run {run_id} | {embedding} - {EMBEDDINGS[embedding]}")
    print(f"  vocab {len(vocab):,} x {matrix.shape[1]} | train UNK rate {rate:.4f} "
          f"({unks:,}/{total:,})")
    print(f"  device {device} | per-device batch {batch_size} | "
          f"embeddings {'fine-tuned' if unfreeze else 'FROZEN'}")

    model = BiLSTMClassifier(matrix, num_classes=2, hidden_dim=hidden_dim,
                             dropout=dropout, freeze_embeddings=not unfreeze).to(device)

    # The split is ~1:4 and macro-F1 is the metric, so an unweighted loss would
    # optimise the wrong thing.
    weights = class_weights(frames["train"]["label"].values).to(device)
    criterion = nn.CrossEntropyLoss(weight=weights)
    optimiser = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr)

    def evaluate(loader):
        model.eval()
        trues, preds, scores = [], [], []
        with torch.no_grad():
            for ids, lengths, labels in loader:
                logits = model(ids.to(device), lengths.to(device))
                preds.append(logits.argmax(dim=1).cpu().numpy())
                scores.append(torch.softmax(logits, dim=1)[:, 1].cpu().numpy())
                trues.append(labels.numpy())
        return (np.concatenate(trues), np.concatenate(preds), np.concatenate(scores))

    best = {"dev_macro_f1": -1.0, "epoch": 0, "state": None}
    history, t0 = [], time.time()

    for epoch in range(1, epochs + 1):
        model.train()
        running = batches = 0
        for ids, lengths, labels in loaders["train"]:
            optimiser.zero_grad()
            loss = criterion(model(ids.to(device), lengths.to(device)), labels.to(device))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimiser.step()
            running += loss.item()
            batches += 1

        y_true, y_pred, _ = evaluate(loaders["dev"])
        dev_f1 = macro_f1(y_true, y_pred)
        history.append({"epoch": epoch, "train_loss": running / batches,
                        "dev_macro_f1": dev_f1})

        marker = ""
        if dev_f1 > best["dev_macro_f1"]:
            best = {"dev_macro_f1": dev_f1, "epoch": epoch,
                    "state": {k: v.detach().cpu().clone()
                              for k, v in model.state_dict().items()}}
            marker = "  <- best"
        print(f"  epoch {epoch:>2}  loss {running / batches:.4f}  "
              f"dev macro-F1 {dev_f1:.4f}{marker}")

        if epoch - best["epoch"] >= PATIENCE:
            print(f"  early stop: {PATIENCE} epochs without improvement")
            break

    train_secs = time.time() - t0

    # Test is scored ONCE, on the checkpoint dev chose.
    model.load_state_dict(best["state"])
    y_true, y_pred, scores = evaluate(loaders["test"])
    metrics = classification_metrics(y_true, y_pred, scores)
    metrics.update({"dev_macro_f1": best["dev_macro_f1"], "best_epoch": best["epoch"],
                    "epochs_run": len(history), "train_seconds": round(train_secs, 1),
                    "train_unk_rate": round(rate, 5)})

    print(f"\n  TEST macro-F1 {metrics['macro_f1']:.4f} | ADE F1 {metrics['f1_ade']:.4f} "
          f"| PR-AUC {metrics['pr_auc']:.3f} | best epoch {best['epoch']} | {train_secs:.0f}s")

    out = OUT / f"run{run_id}_{embedding}"
    out.mkdir(parents=True, exist_ok=True)
    torch.save({"state_dict": best["state"], "config": model.config,
                "embedding": embedding, "run_id": run_id}, out / "checkpoint.pt")
    np.savez(out / "test_predictions.npz", y_true=y_true, y_pred=y_pred, score=scores)
    (out / "metrics.json").write_text(
        json.dumps({"metrics": metrics, "history": history}, indent=2), encoding="utf-8")

    log_run(run_id=run_id, stage="1", model="bilstm_attn", embedding=embedding,
            metrics=metrics,
            params={**model.config, "max_len": MAX_LEN, "patience": PATIENCE,
                    "optimiser": "adam", "grad_clip": 5.0, "class_weights": True,
                    "early_stop_on": "dev macro_f1", "torch": torch.__version__},
            seed=seed, dataset_version=dataset_version,
            per_device_batch=batch_size, device_count=device_count,
            epochs=len(history), lr=lr,
            notes=(f"Step 4.3 run {run_id}; {EMBEDDINGS[embedding]}; embeddings "
                   f"{'fine-tuned' if unfreeze else 'frozen'}; best epoch {best['epoch']}."))
    print(f"  wrote {out}")

**Runs 3-6 are four calls to that one function.** `embedding` is the only argument that
changes — same seed, same hyperparameters, same split, same device count. That is what
makes the result in 4.3 a controlled comparison rather than four separate experiments.

Runs 3u-6u repeat all four with `unfreeze=True`, so the task can update the vectors.

In [ ]:
ABLATION = [("3", "E0_random"), ("4", "E1"), ("5", "E2"), ("6", "E3")]

if TRAIN:
    pin_single_gpu()                     # must happen before torch is imported

    for run_id, embedding in ABLATION:                       # runs 3-6, frozen
        train_bilstm_run(run_id, embedding)

    for run_id, embedding in ABLATION:                       # runs 3u-6u, fine-tuned
        train_bilstm_run(f"{run_id}u", embedding, unfreeze=True)

### T4-T5: the transformers

The same domain-versus-general question one tier up: identical architecture, different
pretraining corpus.

The batch size needs care here. `Trainer` takes a **per-device** batch and multiplies it by
the visible device count, so a recipe saying "batch 16" silently becomes 32 on Kaggle's two
T4s. `derive_per_device_batch` inverts that — you state the effective batch you want, and
the per-device value is computed from the hardware actually present.

In [ ]:
TRANSFORMERS = {
    "7": ("bert-base-uncased", "general English"),
    "8": ("microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract", "pretrained on PubMed"),
}


def train_transformer_run(run_id, *, epochs=3, effective_batch=16, lr=2e-5,
                          max_len=128, seed=DEFAULT_SEED, dataset_version=""):
    """Runs 7 and 8. Same recipe, different pretrained checkpoint."""
    import time

    import numpy as np
    import torch
    from datasets import Dataset
    from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                              DataCollatorWithPadding, Trainer, TrainingArguments)

    from src.metrics import classification_metrics, macro_f1
    from src.utils import derive_per_device_batch

    checkpoint, description = TRANSFORMERS[run_id]
    set_seed(seed)
    device_count = get_device_count()
    per_device = derive_per_device_batch(effective_batch, device_count)

    print(f"run {run_id} | {checkpoint}")
    print(f"  {description} | device_count {device_count} | per-device batch {per_device} "
          f"| effective {per_device * max(device_count, 1)} | lr {lr} | {epochs} epochs")

    tokenizer = AutoTokenizer.from_pretrained(checkpoint)
    frames, datasets = {}, {}
    for split in ("train", "dev", "test"):
        df = pd.read_parquet(SPLITS / f"stage1_{split}.parquet")
        frames[split] = df
        datasets[split] = Dataset.from_pandas(
            df[["text", "label"]], preserve_index=False
        ).map(lambda b: tokenizer(b["text"], truncation=True, max_length=max_len),
              batched=True, remove_columns=["text"])

    truncated = sum(len(tokenizer(t, truncation=False)["input_ids"]) > max_len
                    for t in frames["train"]["text"])
    print(f"  train {len(frames['train']):,} | {truncated} sentences exceed "
          f"max_len={max_len} ({truncated / len(frames['train']):.2%})")

    model = AutoModelForSequenceClassification.from_pretrained(
        checkpoint, num_labels=2,
        id2label={0: "not_ade", 1: "ade"}, label2id={"not_ade": 0, "ade": 1})

    out = OUT / f"run{run_id}_{'bert' if run_id == '7' else 'biomedbert'}"
    args = TrainingArguments(
        output_dir=str(out / "trainer"),
        num_train_epochs=epochs,
        per_device_train_batch_size=per_device,
        per_device_eval_batch_size=per_device * 4,
        learning_rate=lr,
        fp16=torch.cuda.is_available(),
        seed=seed, data_seed=seed,
        eval_strategy="epoch", save_strategy="epoch",
        load_best_model_at_end=True,             # dev chooses the epoch
        metric_for_best_model="macro_f1", greater_is_better=True,
        save_total_limit=1, logging_steps=100, report_to=[])

    trainer = Trainer(
        model=model, args=args,
        train_dataset=datasets["train"], eval_dataset=datasets["dev"],
        data_collator=DataCollatorWithPadding(tokenizer),
        compute_metrics=lambda ep: {"macro_f1": macro_f1(ep.label_ids,
                                                         ep.predictions.argmax(-1))})

    t0 = time.time()
    trainer.train()
    train_secs = time.time() - t0

    dev_f1 = trainer.evaluate(datasets["dev"])["eval_macro_f1"]

    # Test is scored once, on the epoch dev chose.
    output = trainer.predict(datasets["test"])
    logits = torch.as_tensor(output.predictions, dtype=torch.float32)
    scores = torch.softmax(logits, dim=1)[:, 1].numpy()
    y_pred = output.predictions.argmax(-1)

    metrics = classification_metrics(output.label_ids, y_pred, scores)
    metrics.update({"dev_macro_f1": float(dev_f1),
                    "train_seconds": round(train_secs, 1),
                    "truncated_train_sentences": int(truncated)})

    print(f"\n  TEST macro-F1 {metrics['macro_f1']:.4f} | ADE F1 {metrics['f1_ade']:.4f} "
          f"| PR-AUC {metrics['pr_auc']:.3f} | {train_secs:.0f}s")

    trainer.save_model(str(out / "best"))
    tokenizer.save_pretrained(str(out / "best"))
    np.savez(out / "test_predictions.npz",
             y_true=output.label_ids, y_pred=y_pred, score=scores)

    log_run(run_id=run_id, stage="1",
            model="bert-base-uncased" if run_id == "7" else "biomedbert",
            embedding=checkpoint, metrics=metrics,
            params={"max_len": max_len, "fp16": torch.cuda.is_available(),
                    "early_stop_on": "dev macro_f1",
                    "transformers": __import__("transformers").__version__},
            seed=seed, dataset_version=dataset_version,
            per_device_batch=per_device, device_count=device_count,
            epochs=epochs, lr=lr,
            notes=f"Step 4.4 run {run_id}; {checkpoint}; {description}.")
    print(f"  wrote {out / 'best'}")


if TRAIN:
    for run_id in TRANSFORMERS:
        train_transformer_run(run_id)

Run 8 is the same recipe as run 7 with a different starting checkpoint — so the difference
between them is *pretraining domain*, the same question the E1-vs-E2/E3 ablation asks, at a
different scale.

> **Would rerunning this reproduce the exact numbers?** Close, but do not promise identical.
> `set_seed` fixes every RNG and sets `cudnn.deterministic = True`, so the same code on the
> same GPU model should land on the same figures. A different GPU, or a different cuDNN or
> PyTorch version, can move the last decimal place. The ablation *ordering* — E0 < E1 < E2 <
> E3 — is a much larger effect than that and is what the project claims.

---

## 4.3 Results

Every run appended one row to `results/runs.csv` through `log_run` when it finished — score,
hyperparameters, seed, git commit, GPU count. Nothing below is typed by hand; the table is
read straight out of that log.

In [ ]:
runs = pd.read_csv(ROOT / "results" / "runs.csv")
stage1 = runs[runs["stage"].astype(str) == "1"].drop_duplicates("run_id", keep="last")

TIERS = {"1": "T1 counts", "2": "T2 linear", "2b": "T2 linear",
         "3": "T3 BiLSTM", "4": "T3 BiLSTM", "5": "T3 BiLSTM", "6": "T3 BiLSTM",
         "3u": "T3 BiLSTM", "4u": "T3 BiLSTM", "5u": "T3 BiLSTM", "6u": "T3 BiLSTM",
         "7": "T4 BERT", "8": "T5 BiomedBERT"}

table = pd.DataFrame({
    "run": stage1["run_id"],
    "tier": stage1["run_id"].map(TIERS),
    "model": stage1["model"],
    "embedding": stage1["embedding"].str.replace("microsoft/BiomedNLP-", "", regex=False),
    "macro-F1 (test)": stage1["macro_f1"].round(4),
}).set_index("run")
table.loc[[r for r in ["1", "2", "2b", "3", "4", "5", "6", "7", "8"] if r in table.index]]

### The headline: runs 3, 4, 5, 6

Same architecture. Same hyperparameters. Same seed. Same split. **The embedding matrix is
the only thing that changed.**

In [ ]:
ablation = table.loc[["3", "4", "5", "6"]].copy()
ablation["what it is"] = ["random (floor)", "GloVe, general English",
                          "Word2Vec, our PubMed corpus", "FastText, our PubMed corpus"]
floor = ablation.loc["3", "macro-F1 (test)"]
ablation["gain over random"] = (ablation["macro-F1 (test)"] - floor).round(4)
ablation[["embedding", "what it is", "macro-F1 (test)", "gain over random"]]

In [ ]:
from IPython.display import Image
Image(filename=str(ROOT / "results" / "figures" / "stage1_embeddings.png"))

**0.744 → 0.879.** That climb is the project's result.

Read it in three steps:

1. **E0 → E1 (+0.050).** General-purpose English vectors help. Knowing that *the*, *after*
   and *patient* are words is worth something.
2. **E1 → E2 (+0.067).** Training on 159,975 medical abstracts instead of Wikipedia helps
   *more than having vectors at all did*. This is the finding.
3. **E2 → E3 (+0.018).** FastText's character n-grams add a bit on top, consistent with
   notebook 3: it handles the morphological variants of drug names that Word2Vec treats as
   unrelated words.

The dashed line on the chart is run 2, TF-IDF logistic regression. It is there on purpose:
a BiLSTM *below* it would mean the network is undertrained, and you can see that at a
glance instead of having to cross-reference a table. E0 sits below it — a BiLSTM with no
word knowledge does worse than bag-of-words. That is the ablation working correctly.

### Frozen vs fine-tuned: a result that goes the other way

Every ablation run was repeated with the embedding layer *unfrozen* — the `unfreeze=True`
calls above. Those are the runs with a `u` suffix. The two conditions answer different
questions:

- **Frozen** — how good are these vectors?
- **Fine-tuned** — does that advantage survive once the task has had its say?

In [ ]:
compare = pd.DataFrame({
    "embedding": ["E0 random", "E1 GloVe", "E2 Word2Vec", "E3 FastText"],
    "frozen": [table.loc[r, "macro-F1 (test)"] for r in ["3", "4", "5", "6"]],
    "fine-tuned": [table.loc[r, "macro-F1 (test)"] for r in ["3u", "4u", "5u", "6u"]],
})
compare["change"] = (compare["fine-tuned"] - compare["frozen"]).round(4)
compare["fine-tuning"] = compare["change"].map(lambda d: "helped" if d > 0 else "hurt")
compare

**Fine-tuning helped the weak embeddings and hurt the good ones.** The effect reverses with
the quality of the starting vectors.

That is what a small training set predicts: 14,628 sentences carry enough signal to improve
random or general-purpose vectors, but not enough to improve vectors already trained on
159,975 biomedical abstracts. Updating those mostly discards information.

And note where it leaves the ordering: the best fine-tuned run reaches 0.8587 — still below
the *weakest frozen* domain run at 0.8609. Task supervision does not close the gap. That is
the stronger version of the claim, because it says the advantage lives in the vectors
themselves rather than in the head start they provide.

### The transformers

| Run | Model | Macro-F1 | Pretrained on |
|---|---|---|---|
| 7 | `bert-base-uncased` | 0.9159 | general English |
| 8 | **BiomedBERT** | **0.9402** | PubMed abstracts |

The same pattern one tier up: the domain-pretrained model wins by 0.024. Run 8 is the best
Stage 1 model in the project, and it is the gate used for the end-to-end pipeline in
notebook 6.

---

## 4.4 Running the saved model

This is what runs with `TRAIN = False`. The checkpoints written by
`train_bilstm_run` are in `models/stage1/`, and the cell below loads run 6 and scores it on
the test split — on this laptop's CPU, in a couple of seconds.

In [ ]:
import time
from sklearn.metrics import classification_report, f1_score
from src.pipeline import BiLSTMGate

t0 = time.perf_counter()
gate = BiLSTMGate(OUT / "run6_E3" / "checkpoint.pt")
print(f"checkpoint loaded in {time.perf_counter() - t0:.1f}s")

t0 = time.perf_counter()
predictions = [int(is_ade) for _, is_ade in gate.classify(test["text"].tolist())]
print(f"{len(test):,} sentences classified in {time.perf_counter() - t0:.1f}s\n")

print(classification_report(test["label"], predictions, target_names=["not ADE", "ADE"],
                            digits=4))
print(f"macro-F1: {f1_score(test['label'], predictions, average='macro'):.4f}"
      f"   (logged for run 6: {table.loc['6', 'macro-F1 (test)']})")

The number matches the logged one exactly — the checkpoint on this laptop is the same model
that ran on the GPU, and `src/pipeline.py` reassembles it correctly from the saved config.

The per-class breakdown shows where the difficulty is. Precision and recall on `not ADE` are
both around 0.95; on `ADE` they are around 0.80. The minority class is harder, which is
exactly why macro-F1 is the metric and accuracy is not.

In [ ]:
for sentence in ["A case of toxic hepatitis caused by methotrexate.",
                 "The patient was treated with cisplatin, vinblastine and bleomycin.",
                 "No adverse reaction to the vaccine was observed.",
                 "Severe rhabdomyolysis developed after starting simvastatin."]:
    p_ade, is_ade = gate.classify([sentence])[0]
    print(f"{'ADE    ' if is_ade else 'not ADE'}  p={p_ade:.3f}   {sentence}")

---

## What this notebook produced

| Artefact | Contents |
|---|---|
| `models/stage1/run{1..8}*/` | eight trained Stage 1 models |
| `results/runs.csv` | one row per run, written by `log_run` |
| `results/figures/stage1_embeddings.png` | the ablation chart |

**Key numbers:** ablation 0.744 → 0.879 macro-F1; best overall BiomedBERT at 0.940.

### To retrain on Kaggle

1. Start a GPU session and attach the repo plus a Dataset holding
   `models/emb_matrices/*.npy` (57 MB, gitignored — notebook 3 builds them).
2. Set `TRAIN = True` in section 4.2.
3. Run all. Expect a couple of hours for runs 3-8.
4. `models/stage1/` and the new `results/runs.csv` rows are the outputs — Kaggle
   discards `/kaggle/working` when the session ends, so download them before it does.

**Next:** [5 — Stage 2](05_stage2_tagging.ipynb), which finds the drug and the effect inside
the sentences Stage 1 accepts.